<a href="https://colab.research.google.com/github/melissa-04/tubitak-2209a-spatial-stemness-emt/blob/main/02_scrna_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, tarfile
from google.colab import drive
drive.mount('/content/drive')

raw_dir = '/content/drive/MyDrive/Tubitak-2209a/00_raw_data'
scrna_archive = os.path.join(raw_dir, 'GSE176078_Wu_etal_2021_BRCA_scRNASeq.tar.gz')
extract_dir = '/content/scrna_raw'
os.makedirs(extract_dir, exist_ok=True)

with tarfile.open(scrna_archive, 'r:gz') as tar:
    tar.extractall(path=extract_dir, filter='data')

print('Extracted to:', extract_dir)
print('\nContents:')
for root, _, files in os.walk(extract_dir):
    for name in sorted(files):
        p = os.path.join(root, name)
        print(f'  {os.path.relpath(p, extract_dir)}  ({os.path.getsize(p)/1e6:.1f} MB)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracted to: /content/scrna_raw

Contents:
  Wu_etal_2021_BRCA_scRNASeq/count_matrix_barcodes.tsv  (2.5 MB)
  Wu_etal_2021_BRCA_scRNASeq/count_matrix_genes.tsv  (0.3 MB)
  Wu_etal_2021_BRCA_scRNASeq/count_matrix_sparse.mtx  (2395.4 MB)
  Wu_etal_2021_BRCA_scRNASeq/metadata.csv  (11.0 MB)


In [3]:
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        importlib.import_module(name)
        print(f'{name}: already installed')
    except ImportError:
        print(f'{name}: installing...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
        print(f'{name}: done')

ensure('scanpy')
ensure('anndata')

import scanpy as sc
import anndata as ad
print('\nscanpy version:', sc.__version__)
print('anndata version:', ad.__version__)

scanpy: installing...
scanpy: done
anndata: already installed

scanpy version: 1.12.1
anndata version: 0.12.18


/tmp/ipykernel_13286/1280106713.py:18: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print('\nscanpy version:', sc.__version__)
/tmp/ipykernel_13286/1280106713.py:19: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print('anndata version:', ad.__version__)


In [4]:
import scanpy as sc

data_dir = '/content/scrna_raw/Wu_etal_2021_BRCA_scRNASeq'

adata_raw = sc.read_mtx(f'{data_dir}/count_matrix_sparse.mtx')

print('Raw matrix shape (rows, cols):', adata_raw.shape)
print('Matrix dtype:', adata_raw.X.dtype)
print('Is sparse:', type(adata_raw.X))

Raw matrix shape (rows, cols): (29733, 100064)
Matrix dtype: float32
Is sparse: <class 'scipy.sparse._csr.csr_matrix'>


In [5]:
import pandas as pd

adata = adata_raw.T

barcodes = pd.read_csv(f'{data_dir}/count_matrix_barcodes.tsv', header=None)[0].values
genes = pd.read_csv(f'{data_dir}/count_matrix_genes.tsv', header=None)[0].values

adata.obs_names = barcodes
adata.var_names = genes

print('After transpose (cells, genes):', adata.shape)
print('n cells matches barcodes:', adata.n_obs == len(barcodes))
print('n genes matches genes:', adata.n_vars == len(genes))
print('First 3 barcodes:', list(adata.obs_names[:3]))
print('First 3 genes:', list(adata.var_names[:3]))

After transpose (cells, genes): (100064, 29733)
n cells matches barcodes: True
n genes matches genes: True
First 3 barcodes: ['CID3586_AAGACCTCAGCATGAG', 'CID3586_AAGGTTCGTAGTACCT', 'CID3586_ACCAGTAGTTGTGGCC']
First 3 genes: ['RP11-34P13.7', 'FO538757.3', 'FO538757.2']


In [6]:
meta = pd.read_csv(f'{data_dir}/metadata.csv', index_col=0)

print('Metadata shape:', meta.shape)
print('\nColumns:', meta.columns.tolist())
print('\nFirst 3 metadata index (should look like barcodes):')
print(meta.index[:3].tolist())

print('\nSame order as matrix barcodes?',
      list(meta.index[:100]) == list(adata.obs_names[:100]))
print('Same set of barcodes (ignoring order)?',
      set(meta.index) == set(adata.obs_names))
print('Metadata rows == matrix cells?', meta.shape[0] == adata.n_obs)

print('\nPreview:')
meta.head(3)

Metadata shape: (100064, 8)

Columns: ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major']

First 3 metadata index (should look like barcodes):
['CID3586_AAGACCTCAGCATGAG', 'CID3586_AAGGTTCGTAGTACCT', 'CID3586_ACCAGTAGTTGTGGCC']

Same order as matrix barcodes? True
Same set of barcodes (ignoring order)? True
Metadata rows == matrix cells? True

Preview:


,orig.ident,nCount_RNA,nFeature_RNA,percent.mito,subtype,celltype_subset,celltype_minor,celltype_major
CID3586_AAGACCTCAGCATGAG,CID3586,4581,1689,1.506221,HER2+,Endothelial ACKR1,Endothelial ACKR1,Endothelial
CID3586_AAGGTTCGTAGTACCT,CID3586,1726,779,5.793743,HER2+,Endothelial ACKR1,Endothelial ACKR1,Endothelial
CID3586_ACCAGTAGTTGTGGCC,CID3586,1229,514,1.383238,HER2+,Endothelial ACKR1,Endothelial ACKR1,Endothelial


In [7]:
meta_aligned = meta.loc[adata.obs_names]
assert list(meta_aligned.index) == list(adata.obs_names), "Alignment failed!"

adata.obs = meta_aligned.copy()

print('Final AnnData:')
print(adata)
print('\nobs columns now:', adata.obs.columns.tolist())
print('\ncelltype_major counts:')
print(adata.obs['celltype_major'].value_counts())

Final AnnData:
AnnData object with n_obs × n_vars = 100064 × 29733
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major'

obs columns now: ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major']

celltype_major counts:
celltype_major
T-cells              35214
Cancer Epithelial    24489
Myeloid               9675
Endothelial           7605
CAFs                  6573
PVL                   5423
Normal Epithelial     4355
Plasmablasts          3524
B-cells               3206
Name: count, dtype: int64


In [8]:
print('Quality metrics summary (no filtering, just inspection):')
print(adata.obs[['nCount_RNA', 'nFeature_RNA', 'percent.mito']].describe())

print('\n--- Checking Wu et al. thresholds hold ---')
print('Max percent.mito (expect <= ~20):', round(adata.obs['percent.mito'].max(), 2))
print('Min nFeature_RNA (expect >= ~200):', int(adata.obs['nFeature_RNA'].min()))

n_high_mito = (adata.obs['percent.mito'] > 20).sum()
n_low_genes = (adata.obs['nFeature_RNA'] < 200).sum()
print('\nCells above 20% mito:', n_high_mito)
print('Cells below 200 genes:', n_low_genes)

Quality metrics summary (no filtering, just inspection):
          nCount_RNA   nFeature_RNA   percent.mito
count  100064.000000  100064.000000  100064.000000
mean     7139.313369    1778.802926       6.069548
std      9947.078634    1345.232055       3.957157
min       252.000000     201.000000       0.000000
25%      1922.000000     818.000000       3.180594
50%      3573.000000    1339.000000       5.145419
75%      7962.000000    2334.000000       8.095326
max    181558.000000   10358.000000      19.996697

--- Checking Wu et al. thresholds hold ---
Max percent.mito (expect <= ~20): 20.0
Min nFeature_RNA (expect >= ~200): 201

Cells above 20% mito: 0
Cells below 200 genes: 0


In [9]:
import os

assert adata.shape == (100064, 29733), f"Unexpected shape: {adata.shape}"
assert 'celltype_major' in adata.obs.columns, "celltype_major missing"
assert list(adata.obs_names) == list(adata.obs.index), "obs alignment broken"

print('Pre-save checks passed.')
print('Shape:', adata.shape)

output_dir = '/content/drive/MyDrive/Tubitak-2209a/02_scRNA_analysis'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'scrna_filtered.h5ad')

adata.write_h5ad(output_path, compression='gzip')

size_mb = os.path.getsize(output_path) / 1e6
print('\nSAVED:', output_path)
print(f'File size on disk: {size_mb:.1f} MB')

Pre-save checks passed.
Shape: (100064, 29733)

SAVED: /content/drive/MyDrive/Tubitak-2209a/02_scRNA_analysis/scrna_filtered.h5ad
File size on disk: 358.6 MB


In [10]:
import anndata as ad

reloaded = ad.read_h5ad(output_path)

print('Reloaded AnnData:')
print(reloaded)

print('\n--- Integrity checks vs original ---')
print('Shape matches:', reloaded.shape == adata.shape)
print('obs columns match:', list(reloaded.obs.columns) == list(adata.obs.columns))
print('Same cell names:', list(reloaded.obs_names[:50]) == list(adata.obs_names[:50]))
print('Same gene names:', list(reloaded.var_names[:50]) == list(adata.var_names[:50]))

import numpy as np
print('Matrix sum identical:', np.isclose(reloaded.X.sum(), adata.X.sum()))

print('\ncelltype_major counts (should match earlier):')
print(reloaded.obs['celltype_major'].value_counts())

Reloaded AnnData:
AnnData object with n_obs × n_vars = 100064 × 29733
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major'

--- Integrity checks vs original ---
Shape matches: True
obs columns match: True
Same cell names: True
Same gene names: True
Matrix sum identical: True

celltype_major counts (should match earlier):
celltype_major
T-cells              35214
Cancer Epithelial    24489
Myeloid               9675
Endothelial           7605
CAFs                  6573
PVL                   5423
Normal Epithelial     4355
Plasmablasts          3524
B-cells               3206
Name: count, dtype: int64
